In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from prophet import Prophet
import joblib

daily_sales = pd.read_csv(
    r'../data/processed/daily_sales.csv'
)
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

# aggregate to WEEKLY sales
daily_sales['week_start'] = daily_sales['date']\
    .dt.to_period('W').dt.start_time

weekly = daily_sales.groupby('week_start')\
    ['sales'].sum().reset_index()

weekly.columns = ['ds', 'y']
weekly = weekly[weekly['y'] > 0].copy()

print(f"Weekly data shape: {weekly.shape}")
print("Weekly stats:")
print(weekly['y'].describe().round(0))

# time split — last 8 weeks = test
train_w = weekly.iloc[:-8]
test_w  = weekly.iloc[-8:]

print(f"\nTrain weeks: {len(train_w)}")
print(f"Test weeks:  {len(test_w)}")

# train Prophet on weekly data
model_w = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.1
)
model_w.fit(train_w)

# forecast
future_w = model_w.make_future_dataframe(
    periods=8, freq='W'
)
forecast_w = model_w.predict(future_w)

# evaluate
pred_w   = np.clip(
    forecast_w.tail(8)['yhat'].values, 0, None
)
actual_w = test_w['y'].values
mask_w   = actual_w > 0

mape_w = mean_absolute_percentage_error(
    actual_w[mask_w], pred_w[mask_w]
)

print(f"\n=== Weekly Prophet Results ===")
print(f"Weekly MAPE: {mape_w:.1%}")
print(f"Target: ≤ 12%")
print(f"Status: {'✓ PASSES' if mape_w<=0.12 else f'Actual: {mape_w:.1%}'}")

print(f"\nWeek by week:")
for i, (_, row) in enumerate(test_w.iterrows()):
    p = max(0, pred_w[i])
    a = row['y']
    err = abs(a-p)/a*100 if a > 0 else 0
    print(f"  {row['ds'].date()} → "
          f"actual=£{a:,.0f} "
          f"pred=£{p:,.0f} "
          f"err={err:.1f}%")

# save weekly forecast for dashboard
future_12w = model_w.make_future_dataframe(
    periods=12, freq='W'
)
forecast_12w = model_w.predict(future_12w)

forecast_12w.to_csv(
    r'../data/processed/demand_forecast.csv',
    index=False
)

joblib.dump(
    model_w,
    r'../models/prophet_model.pkl'
)

print(f"\n✓ Weekly Prophet model saved")
print(f"✓ Weekly forecast saved")

Weekly data shape: (104, 2)
Weekly stats:
count       104.0
mean     170610.0
std       69587.0
min       35778.0
25%      124568.0
50%      150994.0
75%      202046.0
max      412155.0
Name: y, dtype: float64

Train weeks: 96
Test weeks:  8


18:44:16 - cmdstanpy - INFO - Chain [1] start processing
18:44:17 - cmdstanpy - INFO - Chain [1] done processing



=== Weekly Prophet Results ===
Weekly MAPE: 13.0%
Target: ≤ 12%
Status: Actual: 13.0%

Week by week:
  2011-10-17 → actual=£255,462 pred=£237,718 err=6.9%
  2011-10-24 → actual=£237,685 pred=£241,503 err=1.6%
  2011-10-31 → actual=£260,629 pred=£245,413 err=5.8%
  2011-11-07 → actual=£264,766 pred=£246,895 err=6.7%
  2011-11-14 → actual=£277,774 pred=£263,167 err=5.3%
  2011-11-21 → actual=£249,533 pred=£302,804 err=21.3%
  2011-11-28 → actual=£251,788 pred=£337,247 err=33.9%
  2011-12-05 → actual=£408,587 pred=£317,776 err=22.2%

✓ Weekly Prophet model saved
✓ Weekly forecast saved
